### Document Structure

In [2]:
from langchain_core.documents import Document

In [ ]:
doc=Document(page_content="This is a test document.", 
             metadata={"source": "test.txt",
                       "page": 1,
                       "author": "John Doe",
                       "date_created": "2024-06-01"})

doc

: 

: 

In [ ]:
# Create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)


: 

: 

In [ ]:
sample_text = {
    "../data/text_files/py_sample.txt": 
"""Introduction to Python

Python is a high-level programming language known for its readability and simplicity. It is widely used in web development, data science, automation, and artificial intelligence.

Key Characteristics of Python:

1. Easy to Read
Python uses clear and simple syntax, making it beginner-friendly.

2. Interpreted Language
Python code is executed line by line, which makes debugging easier.

3. Dynamically Typed
You do not need to declare variable types explicitly.

4. Large Standard Library
Python provides many built-in modules for tasks such as file handling, mathematics, and networking.

5. Cross-Platform
Python programs can run on different operating systems with minimal changes.

Common Uses of Python:

- Web development
- Data analysis
- Machine learning
- Automation scripting
- Scientific computing

Conclusion

Python is a versatile and powerful language suitable for both beginners and professionals.""" ,
    "../data/text_files/ml_sample.txt":
"""Introduction to Machine Learning

Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn from data and make predictions or decisions without being explicitly programmed. Instead of following fixed rules, machine learning models identify patterns in data and use them to generate outputs.
Machine learning relies on data, algorithms, and evaluation. Data is used to train the model, algorithms define how learning happens, and evaluation measures how well the model performs on new data.

Types of Machine Learning

1. Supervised Learning
Supervised learning uses labeled data, meaning each input has a corresponding correct output. The model learns to map inputs to outputs. Common tasks include classification (predicting categories) and regression (predicting continuous values).

2. Unsupervised Learning
Unsupervised learning works with unlabeled data. The model tries to discover hidden patterns or structures. Common techniques include clustering and dimensionality reduction.

3. Reinforcement Learning
Reinforcement learning involves an agent that learns by interacting with an environment. The agent receives rewards or penalties based on its actions and aims to maximize total rewards over time.

Conclusion

Machine learning is widely used in real-world applications such as recommendation systems, fraud detection, healthcare, and autonomous systems. Understanding its types helps in choosing the right approach for different problems.

"""
}

for filepath, content in sample_text.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

: 

: 

In [ ]:
# TextLoader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/py_sample.txt")
document = loader.load()
print(document)

: 

: 

In [ ]:
# Directory Loader

from langchain_community.document_loaders import DirectoryLoader

### load all txt files in the directory
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False)

documents = dir_loader.load()
documents

: 

: 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

### load all txt files in the directory
dir_loader = DirectoryLoader(
    "../data/pdf", 
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False)

pdf_documents = dir_loader.load()
pdf_documents

: 

: 

In [ ]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

: 

: 

In [ ]:
chunks = split_documents(pdf_documents)
chunks

: 

: 

### Embedding n Vector Store DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

: 

: 

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Args:
            model_name (str): The name of the HuggingFace model to use for sentence embedding."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            print(f"Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        Args:
            texts: List of text strings to generate embeddings for.
        Returns:
            A numpy array of shape (len(texts), embedding_dimension) containing the embeddings."""
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embedding.shape}")
        return embedding

# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager



: 

: 

### Vector Store 

In [ ]:
class VectorStore:
    """Manages a vector store using ChromaDB for storing document embeddings and metadata."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initializes the vector store.
        Args:
            collection_name (str): The name of the ChromaDB collection to use.
            persist_directory (str): The directory where the ChromaDB data will be persisted."""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            #create persistent ChromaDBClient
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
            metadata={"description": "Collection for storing PDF document embeddings and metadata"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Adds documents and their embeddings to the vector store.
        Args:
            documents: List of document metadata dictionaries to add to the store.
            embeddings: A numpy array of shape (len(documents), embedding_dimension) containing the embeddings."""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i  # Add document index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        # Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store
        




: 

: 

In [ ]:
chunks

: 

: 

In [ ]:
# convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts

: 

: 

In [ ]:
# Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

## store chunks and embeddings in vector store
vector_store.add_documents(chunks, embeddings)

: 

: 

In [ ]:
class RAGRetriever:
    """Retrieves query based retrieval from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the retriever.
        Args:
            vector_store: The vector store containing document embeddings.
            embedding_manager: Manager to generate query embeddings."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves relevant document chunks from the vector store based on a query.
        Args:
            query: The input query string to search for relevant documents.
            top_k: The number of top results to return based on similarity score.
            score_threshold: Minimum cosine similarity score for a document to be considered relevant.
        Returns:
            A list of dictionaries containing the retrieved documents and their metadata."""
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print(f"Retrieved {len(results['documents'][0])} candidate documents from vector store.")

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity
                    print(f"Document ID: {doc_id}, Similarity Score: {similarity_score:.4f}")

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                        print(f"Retrieved {len(retrieved_docs)} documents after filtering")
                    else:
                        print(f"No documents found")

                    return retrieved_docs

        except Exception as e:
            print(f"Error querying vector store: {e}")
            return []
 
rag_retriever = RAGRetriever(vector_store, embedding_manager)

: 

: 

In [ ]:
rag_retriever

: 

: 

In [ ]:
rag_retriever.retrieve("What is Deep Learning?")

: 

: 

: 

: 